# Fake Jobs – Text-Embeddings für die Ausreißererkennung
- Vergleich der Baseline-Detektoren (iForest, LODA, ECOD, AutoEncoder) über drei Repräsentationen
- Gleiche numerische Basis (`cleaned`); Text einmal weggelassen, einmal per **fastText**, einmal per **Sentence-Transformer** (`semantic_pca30`)
- Beste Hyperparameter aus der README (kein GridSearch); kein MLflow-Tracking

In [1]:
import time
import numpy as np
import pandas as pd
import fasttext
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve, auc, classification_report
from pyod.models.iforest import IForest
from pyod.models.loda import LODA
from pyod.models.ecod import ECOD
from pyod.models.auto_encoder import AutoEncoder

## Daten & gemeinsamer Split
- 3 Preprocessing-Varianten über `row_id` alignt, 70/30 stratifiziert (seed 42) → identisches Test-Set

In [2]:
ds = "fake_jobs"
load = lambda name: pd.read_csv(f"../data/preprocessed/{name}_{ds}.csv").set_index("row_id")

cleaned = load("cleaned")
cleaned_text = load("cleaned_text")
semantic_pca30 = load("semantic_pca30")

common = cleaned.index.intersection(semantic_pca30.index).sort_values()
y = cleaned.loc[common, "fraudulent"].values
print("rows", len(common), "outlier rate", round(y.mean(), 4))

tr_id, te_id = train_test_split(common, test_size=0.3, stratify=y, random_state=42)
y_train = cleaned.loc[tr_id, "fraudulent"].values
y_test = cleaned.loc[te_id, "fraudulent"].values

rows 17880 outlier rate 0.0484


## FastText-Embeddings
- 5 Freitextspalten pro Zeile zu einem Dokument verbunden
- Unsupervised skipgram (dim=100) auf dem Korpus, dann Satzvektor je Zeile, StandardScaler

In [3]:
text_cols = ["title", "company_profile", "description", "requirements", "benefits"]
docs = cleaned_text.loc[common, text_cols].fillna("").apply(lambda r: " ".join(r), axis=1).str.replace(r"[\r\n]+", " ", regex=True)

with open("/tmp/ft_corpus.txt", "w") as f:
    f.write("\n".join(docs.tolist()))

ft = fasttext.train_unsupervised("/tmp/ft_corpus.txt", model="skipgram", dim=100)
emb = np.vstack([ft.get_sentence_vector(t) for t in docs])
print("fasttext embeddings", emb.shape)

Read 6M words
Number of words:  49928
Number of labels: 0
Progress: 100.0% words/sec/thread:   48955 lr:  0.000000 avg.loss:  1.569174 ETA:   0h 0m 0s 14.8% words/sec/thread:   46551 lr:  0.042590 avg.loss:  1.805103 ETA:   0h 0m55s 17.0% words/sec/thread:   46580 lr:  0.041506 avg.loss:  1.793428 ETA:   0h 0m53s 17.3% words/sec/thread:   46577 lr:  0.041353 avg.loss:  1.792762 ETA:   0h 0m53s 18.6% words/sec/thread:   46722 lr:  0.040707 avg.loss:  1.786769 ETA:   0h 0m52s 19.4% words/sec/thread:   46859 lr:  0.040292 avg.loss:  1.784601 ETA:   0h 0m52s 20.8% words/sec/thread:   47074 lr:  0.039625 avg.loss:  1.774181 ETA:   0h 0m50s 21.5% words/sec/thread:   47117 lr:  0.039226 avg.loss:  1.769235 ETA:   0h 0m50s 22.3% words/sec/thread:   47122 lr:  0.038834 avg.loss:  1.765700 ETA:   0h 0m49s 26.5% words/sec/thread:   47381 lr:  0.036735 avg.loss:  1.744335 ETA:   0h 0m46s 26.9% words/sec/thread:   47417 lr:  0.036568 avg.loss:  1.743814 ETA:   0h 0m46s 27.7% words/sec/thread:   474

fasttext embeddings (17880, 100)


In [4]:
emb = StandardScaler().fit_transform(emb)
ft_df = pd.DataFrame(emb, index=common, columns=[f"ft_{i}" for i in range(emb.shape[1])])

## Drei Repräsentationen
- Numerische Basis (`cleaned`, 13 Features) für alle identisch; Unterschied nur in der Text-Repräsentation

In [5]:
num = cleaned.drop(columns=["fraudulent"]).loc[common]
pca = semantic_pca30[[c for c in semantic_pca30.columns if c.startswith("pca_")]].loc[common]

reps = {
    "no_text": num,
    "fasttext": num.join(ft_df),
    "semantic_pca30": num.join(pca),
}
for k, v in reps.items():
    print(f"{k:16s} features={v.shape[1]}")

no_text          features=13
fasttext         features=113
semantic_pca30   features=43


## Detektoren & Evaluation
- Beste Hyperparameter aus der README (Fake Jobs), kein GridSearch
- Pro Modell: AP, AUPRC, AUC-ROC, Classification Report

In [6]:
CONT = round(y.mean(), 4)
detectors = {
    "iforest": (IForest, {"n_estimators": 100, "max_features": 1.0, "contamination": CONT, "random_state": 42}),
    "loda": (LODA, {"n_bins": 20, "n_random_cuts": 200, "contamination": CONT}),
    "ecod": (ECOD, {"contamination": CONT}),
    "autoencoder": (AutoEncoder, {"hidden_neuron_list": [64, 32], "epoch_num": 20, "contamination": CONT, "random_state": 42, "device": "cuda"}),
}

results = []
for rep_name, rep in reps.items():
    Xtr, Xte = rep.loc[tr_id].values, rep.loc[te_id].values
    for det_name, (Model, params) in detectors.items():
        t0 = time.perf_counter()
        model = Model(**params)
        model.fit(Xtr)
        scores = model.decision_function(Xte)
        pred = model.predict(Xte)
        runtime = time.perf_counter() - t0
        ap = average_precision_score(y_test, scores)
        prec, rec, _ = precision_recall_curve(y_test, scores)
        auprc = auc(rec, prec)
        auroc = roc_auc_score(y_test, scores)
        results.append({"pipeline": rep_name, "detector": det_name, "AP": ap, "AUPRC": auprc, "AUC_ROC": auroc})
        print(f"\n=== {rep_name} | {det_name} (feat={rep.shape[1]}, t={runtime:.1f}s) ===")
        print(f"AP={ap:.4f}  AUPRC={auprc:.4f}  AUC-ROC={auroc:.4f}")
        print(classification_report(y_test, pred, target_names=["inlier", "outlier"], digits=4))


=== no_text | iforest (feat=13, t=0.4s) ===
AP=0.0876  AUPRC=0.0858  AUC-ROC=0.6141
              precision    recall  f1-score   support

      inlier     0.9559    0.9547    0.9553      5104
     outlier     0.1316    0.1346    0.1331       260

    accuracy                         0.9150      5364
   macro avg     0.5437    0.5447    0.5442      5364
weighted avg     0.9159    0.9150    0.9154      5364


=== no_text | loda (feat=13, t=0.2s) ===
AP=0.0705  AUPRC=0.0685  AUC-ROC=0.5646
              precision    recall  f1-score   support

      inlier     0.9542    0.9512    0.9527      5104
     outlier     0.0978    0.1038    0.1007       260

    accuracy                         0.9101      5364
   macro avg     0.5260    0.5275    0.5267      5364
weighted avg     0.9127    0.9101    0.9114      5364


=== no_text | ecod (feat=13, t=0.5s) ===
AP=0.0759  AUPRC=0.0745  AUC-ROC=0.6125
              precision    recall  f1-score   support

      inlier     0.9557    0.9587    0.957

Training: 100%|██████████| 20/20 [00:28<00:00,  1.42s/it]



=== no_text | autoencoder (feat=13, t=35.1s) ===
AP=0.0838  AUPRC=0.0816  AUC-ROC=0.6003
              precision    recall  f1-score   support

      inlier     0.9540    0.9516    0.9528      5104
     outlier     0.0952    0.1000    0.0976       260

    accuracy                         0.9103      5364
   macro avg     0.5246    0.5258    0.5252      5364
weighted avg     0.9124    0.9103    0.9114      5364


=== fasttext | iforest (feat=113, t=0.4s) ===
AP=0.0881  AUPRC=0.0871  AUC-ROC=0.6773
              precision    recall  f1-score   support

      inlier     0.9533    0.9589    0.9560      5104
     outlier     0.0870    0.0769    0.0816       260

    accuracy                         0.9161      5364
   macro avg     0.5201    0.5179    0.5188      5364
weighted avg     0.9113    0.9161    0.9137      5364


=== fasttext | loda (feat=113, t=0.4s) ===
AP=0.0677  AUPRC=0.0670  AUC-ROC=0.6406
              precision    recall  f1-score   support

      inlier     0.9511    0.9

Training: 100%|██████████| 20/20 [00:30<00:00,  1.52s/it]



=== fasttext | autoencoder (feat=113, t=31.4s) ===
AP=0.1177  AUPRC=0.1164  AUC-ROC=0.7410
              precision    recall  f1-score   support

      inlier     0.9574    0.9565    0.9570      5104
     outlier     0.1623    0.1654    0.1638       260

    accuracy                         0.9182      5364
   macro avg     0.5599    0.5609    0.5604      5364
weighted avg     0.9189    0.9182    0.9185      5364


=== semantic_pca30 | iforest (feat=43, t=0.4s) ===
AP=0.0762  AUPRC=0.0751  AUC-ROC=0.5971
              precision    recall  f1-score   support

      inlier     0.9546    0.9555    0.9551      5104
     outlier     0.1098    0.1077    0.1087       260

    accuracy                         0.9144      5364
   macro avg     0.5322    0.5316    0.5319      5364
weighted avg     0.9136    0.9144    0.9140      5364


=== semantic_pca30 | loda (feat=43, t=0.7s) ===
AP=0.0510  AUPRC=0.0504  AUC-ROC=0.5536
              precision    recall  f1-score   support

      inlier     0

Training: 100%|██████████| 20/20 [00:29<00:00,  1.49s/it]



=== semantic_pca30 | autoencoder (feat=43, t=30.8s) ===
AP=0.0610  AUPRC=0.0603  AUC-ROC=0.5848
              precision    recall  f1-score   support

      inlier     0.9535    0.9432    0.9483      5104
     outlier     0.0794    0.0962    0.0870       260

    accuracy                         0.9021      5364
   macro avg     0.5164    0.5197    0.5176      5364
weighted avg     0.9111    0.9021    0.9065      5364



## Ergebnisvergleich
- Mittelwert je Pipeline über alle Detektoren → beste Text-Repräsentation

In [7]:
res = pd.DataFrame(results)
print(res.round(4).to_string(index=False))

summary = res.groupby("pipeline")[["AP", "AUPRC", "AUC_ROC"]].mean().sort_values("AUPRC", ascending=False)
print("\nMittelwert über alle Detektoren:")
print(summary.round(4).to_string())
print("\nBeste Pipeline (nach AUPRC):", summary.index[0])

      pipeline    detector     AP  AUPRC  AUC_ROC
       no_text     iforest 0.0876 0.0858   0.6141
       no_text        loda 0.0705 0.0685   0.5646
       no_text        ecod 0.0759 0.0745   0.6125
       no_text autoencoder 0.0838 0.0816   0.6003
      fasttext     iforest 0.0881 0.0871   0.6773
      fasttext        loda 0.0677 0.0670   0.6406
      fasttext        ecod 0.0759 0.0750   0.6478
      fasttext autoencoder 0.1177 0.1164   0.7410
semantic_pca30     iforest 0.0762 0.0751   0.5971
semantic_pca30        loda 0.0510 0.0504   0.5536
semantic_pca30        ecod 0.0563 0.0557   0.5471
semantic_pca30 autoencoder 0.0610 0.0603   0.5848

Mittelwert über alle Detektoren:
                    AP   AUPRC  AUC_ROC
pipeline                               
fasttext        0.0874  0.0864   0.6767
no_text         0.0795  0.0776   0.5979
semantic_pca30  0.0611  0.0604   0.5706

Beste Pipeline (nach AUPRC): fasttext
